<a href="https://colab.research.google.com/github/mikkaK/ZC3-team-generator/blob/main/ZC3_Team_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import csv
import re
from datetime import datetime

#CSV_PATH = "/content/ZC3_Company_Triathlon_28.06.csv"
#CSV_PATH = "/content/zc3_edgecases_conflicts.csv"
CSV_PATH = "/content/zc3_edgecases_valid.csv"

ROLES = ["swimmer", "biker", "runner"]

def _norm(s: str) -> str:
    return (s or "").strip()

def _norm_lower(s: str) -> str:
    return _norm(s).lower()

def _pick(row: dict, name: str) -> str:
    if name in row:
        return row.get(name, "") or ""
    m = {k.lower(): k for k in row.keys()}
    k = m.get(name.lower())
    return (row.get(k, "") or "") if k else ""

def parse_completion_time(raw: str):
    raw = _norm(raw)
    if not raw:
        return None
    # Your CSV example: "1/30/26 13:15"
    for fmt in ("%m/%d/%y %H:%M", "%m/%d/%Y %H:%M", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try:
            return datetime.strptime(raw, fmt)
        except ValueError:
            pass
    return None

def parse_distance(raw: str) -> str:
    r = _norm_lower(raw)
    if "both" in r:
        return "both"
    if "sprint" in r:
        return "sprint"
    if "olympic" in r:
        return "olympic"
    return "both"

def parse_ambition(raw: str) -> str:
    r = _norm_lower(raw)
    if "competitive" in r:
        return "competitive"
    if "fun" in r:
        return "fun"
    if "both" in r:
        return "both"
    return "both"

def parse_gender(raw: str) -> str:
    r = _norm_lower(raw)
    if r in ("man", "m", "male"):
        return "m"
    if r in ("woman", "w", "f", "female"):
        return "f"
    return "x"

def parse_prio(raw: str):
    r = _norm_lower(raw).replace(" ", "")
    if "nogo" in r or "no-go" in r:
        return "nogo"
    m = re.search(r"prio(\d)", r)
    return int(m.group(1)) if m else None

def load_registrants(csv_path: str) -> list[dict]:
    regs = []
    with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            completed_at = parse_completion_time(_pick(row, "Completion time"))

            # Use Company Email if present, else fallback to Email; normalize to lowercase
            email = _norm_lower(_pick(row, "Company Email (firstname.name@accenture.com)")) or _norm_lower(_pick(row, "Email"))
            if not email:
                continue

            # teammate emails (normalized)
            tm_swim = _norm_lower(_pick(row, "Swimmer (Company Email)"))
            tm_bike = _norm_lower(_pick(row, "Biker (Company Email)"))
            tm_run  = _norm_lower(_pick(row, "Runner (Company Email)"))

            pr_swim = parse_prio(_pick(row, "Priority.Swimmer"))
            pr_bike = parse_prio(_pick(row, "Priority.Biker"))
            pr_run  = parse_prio(_pick(row, "Priority.Runner"))

            prios = {"swimmer": pr_swim, "biker": pr_bike, "runner": pr_run}
            prio_by_rank = {1: None, 2: None, 3: None}
            nogo = []

            for role, pr in prios.items():
                if pr == "nogo":
                    nogo.append(role)
                elif isinstance(pr, int) and pr in (1, 2, 3) and prio_by_rank[pr] is None:
                    prio_by_rank[pr] = role

            remaining = [r for r in ROLES if (r not in nogo and r not in prio_by_rank.values())]
            for rank in (1, 2, 3):
                if prio_by_rank[rank] is None and remaining:
                    prio_by_rank[rank] = remaining.pop(0)

            regs.append({
                "id": email,
                "name": _norm(_pick(row, "Name")),
                "completed_at": completed_at,

                "gender": parse_gender(_pick(row, "Gender")),
                "distance": parse_distance(_pick(row, "Preferred distance")),
                "ambition": parse_ambition(_pick(row, "Ambition")),
                "remarks": _norm(_pick(row, "Remarks")),

                "prio1": prio_by_rank[1] or "swimmer",
                "prio2": prio_by_rank[2] or "biker",
                "prio3": prio_by_rank[3] or "runner",
                "nogo": nogo,

                "requested_teammates": {
                    "swimmer": tm_swim,
                    "biker": tm_bike,
                    "runner": tm_run,
                },
            })

    # Sort chronologically (earliest first). Missing timestamps go last.
    regs.sort(key=lambda r: (r["completed_at"] is None, r["completed_at"]))
    return regs

registrants = load_registrants(CSV_PATH)
print("Loaded:", len(registrants))
print("First 3 (chronological):")
for r in registrants[:3]:
    print(r["completed_at"], r["id"], r["distance"], r["ambition"])

Loaded: 60
First 3 (chronological):
2026-06-28 08:00:00 person001@accenture.com olympic competitive
2026-06-28 08:05:00 person002@accenture.com olympic competitive
2026-06-28 08:10:00 person003@accenture.com olympic competitive


In [ ]:
# ---- Build hard custom groups via Union-Find, with role locks ----

class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra

def build_units(registrants: list[dict]):
    by_id = {r["id"]: r for r in registrants}
    uf = UnionFind()
    issues = []

    for r in registrants:
        uf.find(r["id"])

    # must-link edges
    for r in registrants:
        me = r["id"]
        for role, mate in r["requested_teammates"].items():
            if mate:
                if mate not in by_id:
                    issues.append(f"Teammate not found: {me} requested {role}={mate}")
                    continue
                uf.union(me, mate)

    # groups
    groups = {}
    for pid in by_id:
        root = uf.find(pid)
        groups.setdefault(root, []).append(pid)

    # role locks: if A enters runner=B => B locked to runner in that group
    role_locks_by_group = {root: {} for root in groups}
    for r in registrants:
        root = uf.find(r["id"])
        locks = role_locks_by_group[root]
        for role, mate in r["requested_teammates"].items():
            if mate and mate in by_id:
                if mate in locks and locks[mate] != role:
                    issues.append(f"ROLE CONFLICT: {mate} locked as {locks[mate]} and {role}")
                locks[mate] = role

    used_in_group = set()
    units = []

    for root, member_ids in groups.items():
        if len(member_ids) > 3:
            issues.append(f"GROUP TOO LARGE (>3): {root} has {len(member_ids)} members: {member_ids}")
            continue

        members = [by_id[mid] for mid in member_ids]
        used_in_group.update(member_ids)

        # unit time = earliest completion among members
        times = [m["completed_at"] for m in members if m["completed_at"] is not None]
        unit_time = min(times) if times else None

        units.append({
            "unit_id": root,
            "members": members,
            "role_locks": role_locks_by_group[root],  # member_id -> locked_role
            "is_custom": True,
            "unit_time": unit_time,
        })

    # singles not in any group
    for r in registrants:
        if r["id"] not in used_in_group:
            units.append({
                "unit_id": r["id"],
                "members": [r],
                "role_locks": {},
                "is_custom": False,
                "unit_time": r["completed_at"],
            })

    # chronological ordering (missing timestamps last)
    units.sort(key=lambda u: (u["unit_time"] is None, u["unit_time"]))

    return units, issues

units, issues = build_units(registrants)
print("Units:", len(units))
if issues:
    print("\nISSUES (first 30):")
    for x in issues[:30]:
        print(" -", x)

Units: 56


In [ ]:
import itertools
import csv

TEAM_PLAN = [("olympic", 7), ("sprint", 8)]  # exactly 7 olympic teams + 8 sprint teams

def ambition_hard_ok(people):
    ambs = {p["ambition"] for p in people}
    return not ("competitive" in ambs and "fun" in ambs)

def choose_team_distance(people):
    fixed = set()
    for p in people:
        d = p["distance"]
        if d in ("sprint", "olympic"):
            fixed.add(d)
    if len(fixed) == 0:
        return "free"
    if len(fixed) == 1:
        return next(iter(fixed))
    return None  # impossible

def role_allowed(person, role):
    return role not in set(person.get("nogo", []))

def role_score(person, role):
    if not role_allowed(person, role):
        return -10**9
    if role == person["prio1"]: return 30
    if role == person["prio2"]: return 20
    if role == person["prio3"]: return 10
    return 1

def best_role_assignment(people, role_locks):
    ids = [p["id"] for p in people]
    locked_roles = [role_locks[i] for i in ids if i in role_locks]
    if len(set(locked_roles)) != len(locked_roles):
        return None, None

    best = None
    best_map = None
    for perm in itertools.permutations(ROLES, len(people)):
        ok = True
        total = 0
        m = {}
        for p, role in zip(people, perm):
            pid = p["id"]
            if pid in role_locks and role_locks[pid] != role:
                ok = False; break
            s = role_score(p, role)
            if s < -10**8:
                ok = False; break
            total += s
            m[pid] = role
        if ok and (best is None or total > best):
            best, best_map = total, m
    return best, best_map

def unit_distance_eligibility(unit, target_dist):
    # olympic: members must be olympic or both
    # sprint: members must be sprint or both
    allowed = {target_dist, "both"}
    return all(m["distance"] in allowed for m in unit["members"])

def unit_is_both_only(unit):
    return all(m["distance"] == "both" for m in unit["members"])

def team_score(people, role_locks, target_dist):
    # hard ambition
    if not ambition_hard_ok(people):
        return -10**9, None

    # hard distance
    if any(p["distance"] not in (target_dist, "both") for p in people):
        return -10**9, None

    # role feasibility
    rs, role_map = best_role_assignment(people, role_locks)
    if role_map is None:
        return -10**9, None

    # ambition preference (strong)
    ambs = [p["ambition"] for p in people]
    if all(a == "competitive" for a in ambs): amb_score = 200
    elif ("competitive" in ambs and "fun" not in ambs): amb_score = 140
    elif all(a == "fun" for a in ambs): amb_score = 120
    elif ("fun" in ambs and "competitive" not in ambs): amb_score = 90
    else: amb_score = 0

    # joker penalty
    joker_pen = sum(3 for p in people if p["distance"] == "both")

    return rs + amb_score - joker_pen, role_map

def pick_next_team(units_available, target_dist):
    """
    Build one team of 3 people using chronological priority:
    - anchor = earliest eligible unit
    - then dynamically widen search window until feasible completion is found
    """
    eligible = [u for u in units_available if unit_distance_eligibility(u, target_dist)]
    if not eligible:
        return None, None, None

    # sprint phase rule: prioritize sprint signups first, then both
    if target_dist == "sprint":
        eligible.sort(key=lambda u: (unit_is_both_only(u), u["unit_time"] is None, u["unit_time"]))
    else:
        eligible.sort(key=lambda u: (u["unit_time"] is None, u["unit_time"]))

    anchor = eligible[0]
    anchor_size = len(anchor["members"])

    # try growing window
    window = 30
    step = 25

    while True:
        cand = eligible[:min(window, len(eligible))]

        # Must include anchor
        others = [u for u in cand if u["unit_id"] != anchor["unit_id"]]

        best = None
        best_choice = None
        best_roles = None

        # combinations depending on anchor size
        if anchor_size == 3:
            people = anchor["members"]
            score, roles = team_score(people, dict(anchor["role_locks"]), target_dist)
            if score > -10**8:
                return [anchor], roles, score

        elif anchor_size == 2:
            for u1 in others:
                if len(u1["members"]) != 1:
                    continue
                people = anchor["members"] + u1["members"]
                locks = dict(anchor["role_locks"])
                locks.update(u1["role_locks"])
                score, roles = team_score(people, locks, target_dist)
                if score > -10**8 and (best is None or score > best):
                    best, best_choice, best_roles = score, [anchor, u1], roles

        elif anchor_size == 1:
            # anchor + (size2) OR anchor + (size1+size1)
            for u1 in others:
                if len(u1["members"]) == 2:
                    people = anchor["members"] + u1["members"]
                    locks = dict(anchor["role_locks"]); locks.update(u1["role_locks"])
                    score, roles = team_score(people, locks, target_dist)
                    if score > -10**8 and (best is None or score > best):
                        best, best_choice, best_roles = score, [anchor, u1], roles

            singles = [u for u in others if len(u["members"]) == 1]
            for u1, u2 in itertools.combinations(singles, 2):
                people = anchor["members"] + u1["members"] + u2["members"]
                locks = dict(anchor["role_locks"]); locks.update(u1["role_locks"]); locks.update(u2["role_locks"])
                score, roles = team_score(people, locks, target_dist)
                if score > -10**8 and (best is None or score > best):
                    best, best_choice, best_roles = score, [anchor, u1, u2], roles

        if best_choice is not None:
            return best_choice, best_roles, best

        if window >= len(eligible):
            return None, None, None  # no feasible completion exists
        window = min(len(eligible), window + step)

def build_exact_teams(units, plan):
    units_available = units[:]
    final_teams = []

    for dist, team_count in plan:
        for _ in range(team_count):
            chosen_units, roles, score = pick_next_team(units_available, dist)
            if chosen_units is None:
                raise RuntimeError(f"Could not complete required team for {dist}. Try relaxing constraints or check data conflicts.")
            # remove chosen units from availability
            chosen_ids = {u["unit_id"] for u in chosen_units}
            units_available = [u for u in units_available if u["unit_id"] not in chosen_ids]

            members = []
            role_locks = {}
            for u in chosen_units:
                members.extend(u["members"])
                role_locks.update(u["role_locks"])

            final_teams.append({
                "distance": dist,
                "members": members,
                "roles": roles,
                "team_type": "CUSTOM" if any(u["is_custom"] for u in chosen_units) else "AUTO"
            })

    # waitlist = all remaining members from remaining units
    waitlist = []
    for u in units_available:
        waitlist.extend(u["members"])

    return final_teams, waitlist

final_teams, waitlist = build_exact_teams(units, TEAM_PLAN)

print("Teams built:", len(final_teams), "=> athletes:", sum(len(t["members"]) for t in final_teams))
print("Waitlist athletes:", len(waitlist))

# --- compatibility hint for waitlist ---
def waitlist_hint(person):
    # distance fit
    d = person["distance"]
    if d == "olympic": dist_hint = "olympic"
    elif d == "sprint": dist_hint = "sprint"
    else: dist_hint = "either (both)"
    # ambition fit
    a = person["ambition"]
    if a == "competitive": amb_hint = "competitive teams only"
    elif a == "fun": amb_hint = "fun teams only"
    else: amb_hint = "either (both)"
    # role pref
    return dist_hint, amb_hint, person["prio1"]

# --- export CSV ---
OUTPUT_CSV = "/content/zc3_team_assignment_result.csv"

rows = []
for team_idx, t in enumerate(final_teams, start=1):
    for m in t["members"]:
        rows.append({
            "team_id": team_idx,
            "team_type": t["team_type"],
            "team_distance": t["distance"],
            "completed_at": m["completed_at"].isoformat(sep=" ") if m["completed_at"] else "",
            "member_email": m["id"],
            "member_name": m.get("name",""),
            "gender": m.get("gender",""),
            "ambition": m.get("ambition",""),
            "preferred_distance": m.get("distance",""),
            "assigned_role": t["roles"].get(m["id"], ""),
            "waitlist": "NO",
            "waitlist_distance_hint": "",
            "waitlist_ambition_hint": "",
            "waitlist_best_role": "",
        })

for m in waitlist:
    dist_hint, amb_hint, best_role = waitlist_hint(m)
    rows.append({
        "team_id": "WAITLIST",
        "team_type": "WAITLIST",
        "team_distance": "",
        "completed_at": m["completed_at"].isoformat(sep=" ") if m["completed_at"] else "",
        "member_email": m["id"],
        "member_name": m.get("name",""),
        "gender": m.get("gender",""),
        "ambition": m.get("ambition",""),
        "preferred_distance": m.get("distance",""),
        "assigned_role": "",
        "waitlist": "YES",
        "waitlist_distance_hint": dist_hint,
        "waitlist_ambition_hint": amb_hint,
        "waitlist_best_role": best_role,
    })

with open(OUTPUT_CSV, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Exported:", OUTPUT_CSV)

Teams built: 15 => athletes: 45
Waitlist athletes: 15
Exported: /content/zc3_team_assignment_result.csv


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
from collections import Counter
from datetime import datetime

OUTPUT_XLSX = "/content/zc3_team_assignment_result.xlsx"

def auto_width(ws, min_w=12, max_w=50):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = max(min_w, min(max_w, max_len + 2))

def style_header(ws):
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_font = Font(bold=True, color="FFFFFF")
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.freeze_panes = "A2"

wb = Workbook()
wb.remove(wb.active)

# =====================================================
# Sheet 1 — Teams (flat, Excel-friendly)
# =====================================================
ws = wb.create_sheet("Teams")

headers = [
    "team_id", "team_type", "team_distance",
    "assigned_role", "member_name", "member_email",
    "gender", "ambition", "preferred_distance", "completed_at"
]
ws.append(headers)

for team_idx, team in enumerate(final_teams, start=1):
    for m in team["members"]:
        ws.append([
            team_idx,
            team.get("team_type", "AUTO"),
            team["distance"],
            team["roles"].get(m["id"], ""),
            m.get("name", ""),
            m["id"],
            m.get("gender", ""),
            m.get("ambition", ""),
            m.get("distance", ""),
            m["completed_at"].isoformat(" ") if m.get("completed_at") else "",
        ])

style_header(ws)
auto_width(ws)

# =====================================================
# Sheet 2 — Team Summary (one row per team)
# =====================================================
ws = wb.create_sheet("Team_Summary")
headers = [
    "team_id", "team_type", "distance",
    "members",
    "gender_m", "gender_f", "gender_x",
    "competitive", "fun", "both",
    "swimmer", "biker", "runner",
]
ws.append(headers)

for team_idx, team in enumerate(final_teams, start=1):
    genders = Counter(m.get("gender") for m in team["members"])
    ambitions = Counter(m.get("ambition") for m in team["members"])
    roles = Counter(team["roles"].values())

    ws.append([
        team_idx,
        team.get("team_type", "AUTO"),
        team["distance"],
        ", ".join(m["id"] for m in team["members"]),
        genders.get("m", 0),
        genders.get("f", 0),
        genders.get("x", 0),
        ambitions.get("competitive", 0),
        ambitions.get("fun", 0),
        ambitions.get("both", 0),
        roles.get("swimmer", 0),
        roles.get("biker", 0),
        roles.get("runner", 0),
    ])

style_header(ws)
auto_width(ws)

# =====================================================
# Sheet 3 — Waitlist
# =====================================================
ws = wb.create_sheet("Waitlist")

headers = [
    "completed_at", "email", "name",
    "gender", "ambition", "preferred_distance",
    "best_role", "distance_hint", "ambition_hint"
]
ws.append(headers)

for m in waitlist:
    if m["distance"] == "both":
        dist_hint = "sprint or olympic"
    else:
        dist_hint = m["distance"]

    if m["ambition"] == "both":
        amb_hint = "fun or competitive"
    else:
        amb_hint = m["ambition"]

    ws.append([
        m["completed_at"].isoformat(" ") if m.get("completed_at") else "",
        m["id"],
        m.get("name", ""),
        m.get("gender", ""),
        m.get("ambition", ""),
        m.get("distance", ""),
        m.get("prio1", ""),
        dist_hint,
        amb_hint,
    ])

style_header(ws)
auto_width(ws)

# =====================================================
# Sheet 4 — Raw Input (chronological)
# =====================================================
ws = wb.create_sheet("Raw_Input")

headers = [
    "completed_at", "email", "name",
    "gender", "distance", "ambition",
    "teammate_swimmer", "teammate_biker", "teammate_runner",
    "remarks"
]
ws.append(headers)

for r in registrants:
    ws.append([
        r["completed_at"].isoformat(" ") if r.get("completed_at") else "",
        r["id"],
        r.get("name", ""),
        r.get("gender", ""),
        r.get("distance", ""),
        r.get("ambition", ""),
        r["requested_teammates"].get("swimmer", ""),
        r["requested_teammates"].get("biker", ""),
        r["requested_teammates"].get("runner", ""),
        r.get("remarks", ""),
    ])

style_header(ws)
auto_width(ws)

# =====================================================
# Save
# =====================================================
wb.save(OUTPUT_XLSX)
print(f"✅ Excel export created: {OUTPUT_XLSX}")

✅ Excel export created: /content/zc3_team_assignment_result.xlsx
